# Profiling

> Per-layer profiling for deep analysis of model performance

In [ ]:
#| default_exp profiling

In [ ]:
#| export
from __future__ import annotations

import time
import warnings
from typing import Sequence

import numpy as np
import torch
import torch.nn as nn

from fasterbench.core import _bytes_to_mib, _device_ctx, _section, _fmt_table, _fmt_float, _fmt_macs

In [ ]:
#| export
def _tensor_bytes(t: torch.Tensor) -> int:  # tensor to measure
    """Calculate memory footprint of a tensor in bytes."""
    return t.numel() * t.element_size()


def _output_bytes(output) -> int:  # layer output (tensor, tuple, list, or dict)
    """Calculate total bytes for layer output, handling nested structures."""
    if isinstance(output, torch.Tensor):
        return _tensor_bytes(output)
    elif isinstance(output, (tuple, list)):
        return sum(_output_bytes(o) for o in output)
    elif isinstance(output, dict):
        return sum(_output_bytes(v) for v in output.values())
    return 0


def _leaf_modules(model: nn.Module) -> dict[str, nn.Module]:
    """Return dict of leaf modules (no children) with non-empty names."""
    return {
        name: module
        for name, module in model.named_modules()
        if len(list(module.children())) == 0 and name
    }


#| export
def _setup_size_hooks(
    leaf_modules: dict[str, nn.Module],  # {name: module} for leaf modules
    measurements: dict[str, list],        # {name: []} to store param counts
    device_type: str | None = None,       # unused, for consistent signature
) -> list:
    """Compute parameter counts directly - no actual hooks needed."""
    for name, module in leaf_modules.items():
        params = sum(p.numel() for p in module.parameters(recurse=False))
        measurements[name].append(params)
    return []  # No handles to remove


#| export
def _setup_speed_hooks(
    leaf_modules: dict[str, nn.Module],  # {name: module} for leaf modules
    measurements: dict[str, list],        # {name: []} to store timing measurements (ms)
    device_type: str | None = None,       # "cuda" or "cpu"
) -> list:
    """Create pre+post forward hooks for timing each layer."""
    handles = []
    
    def make_cuda_hooks(layer_name: str):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        def pre_hook(mod, inp):
            start.record()
        def post_hook(mod, inp, output):
            end.record()
            torch.cuda.synchronize()
            measurements[layer_name].append(start.elapsed_time(end))
        return pre_hook, post_hook
    
    def make_cpu_hooks(layer_name: str):
        state = {}
        def pre_hook(mod, inp):
            state['start'] = time.perf_counter() * 1000
        def post_hook(mod, inp, output):
            measurements[layer_name].append(time.perf_counter() * 1000 - state['start'])
        return pre_hook, post_hook
    
    make_hooks = make_cuda_hooks if device_type == "cuda" else make_cpu_hooks
    
    for name, module in leaf_modules.items():
        pre_hook, post_hook = make_hooks(name)
        handles.append(module.register_forward_pre_hook(pre_hook))
        handles.append(module.register_forward_hook(post_hook))
    
    return handles


#| export
def _setup_memory_hooks(
    leaf_modules: dict[str, nn.Module],  # {name: module} for leaf modules
    measurements: dict[str, list],        # {name: []} to store memory measurements (MiB)
    device_type: str | None = None,       # unused, for consistent signature
) -> list:
    """Create forward hooks to measure output tensor size per layer (in MiB)."""
    handles = []
    for name, module in leaf_modules.items():
        def make_hook(layer_name: str):
            def hook(mod, inp, output):
                mib = _bytes_to_mib(_output_bytes(output))
                measurements[layer_name].append(mib)
            return hook
        handles.append(module.register_forward_hook(make_hook(name)))
    return handles


#| export
def _setup_compute_hooks(
    leaf_modules: dict[str, nn.Module],  # {name: module} for leaf modules
    measurements: dict[str, list],        # {name: []} to store MAC measurements
    device_type: str | None = None,       # unused, for consistent signature
) -> list | None:
    """Create forward hooks to count MACs per layer using torchprofile."""
    try:
        from torchprofile.handlers import handlers
    except ImportError:
        warnings.warn("torchprofile not installed. pip install torchprofile")
        return None
    
    hook_handles = []
    for name, module in leaf_modules.items():
        handler = handlers.get(type(module))
        if handler:
            def make_hook(layer_name: str, handler_fn):
                def hook(mod, inp, output):
                    try:
                        macs = handler_fn(mod, inp, output)
                        if macs is not None:
                            measurements[layer_name].append(macs)
                    except Exception:
                        pass
                return hook
            hook_handles.append(module.register_forward_hook(make_hook(name, handler)))
    return hook_handles


#| export
_METRIC_CONFIG = {
    "size": {"col": "params", "needs_inference": False, "aggregate": "sum"},
    "speed": {"col": "time_ms", "needs_inference": True, "aggregate": "mean"},
    "memory": {"col": "memory_mib", "needs_inference": True, "aggregate": "mean"},
    "compute": {"col": "macs", "needs_inference": True, "aggregate": "sum", "single_pass": True},
}

_HOOK_SETUP = {
    "size": _setup_size_hooks,
    "speed": _setup_speed_hooks,
    "memory": _setup_memory_hooks,
    "compute": _setup_compute_hooks,
}


@torch.inference_mode()
def _profile_layers(
    model: nn.Module,              # model to profile
    sample: torch.Tensor,          # input tensor (with batch dimension)
    *,
    device: str | torch.device,    # device to run profiling on
    metric: str,                   # "size", "speed", "memory", or "compute"
    warmup: int = 5,               # warmup iterations
    steps: int = 20,               # measurement iterations
) -> list[dict]:
    """Generic per-layer profiling using forward hooks.
    
    Returns list of dicts with: name, type, value, percent (sorted by value descending).
    """
    if metric not in _METRIC_CONFIG:
        raise ValueError(f"Invalid metric: {metric}. Valid: {list(_METRIC_CONFIG.keys())}")
    
    config = _METRIC_CONFIG[metric]
    col = config["col"]
    
    with _device_ctx(device) as dev:
        model = model.eval().to(dev)
        sample = sample.to(dev)
        
        leaf_mods = _leaf_modules(model)
        
        measurements: dict[str, list] = {name: [] for name in leaf_mods}
        
        # Setup metric-specific hooks (all have same signature now)
        handles = _HOOK_SETUP[metric](leaf_mods, measurements, dev.type)
        if handles is None:  # e.g., torchprofile not available
            return []
        
        try:
            # Run inference if needed
            if config["needs_inference"]:
                # Warmup (skip for single-pass metrics like compute)
                if not config.get("single_pass", False):
                    for _ in range(warmup):
                        model(sample)
                        for name in measurements:
                            measurements[name].clear()
                
                # Measurement
                num_steps = 1 if config.get("single_pass", False) else steps
                for _ in range(num_steps):
                    model(sample)
            
            # Aggregate results
            results = []
            total = 0.0
            
            for name, module in leaf_mods.items():
                if measurements[name]:
                    if config["aggregate"] == "mean":
                        value = float(np.mean(measurements[name]))
                    else:  # sum
                        value = float(sum(measurements[name]))
                else:
                    value = 0.0
                
                total += value
                results.append({
                    "name": name,
                    "type": module.__class__.__name__,
                    col: value,
                })
            
            # Add percentages
            for r in results:
                r["percent"] = (r[col] / total * 100) if total > 0 else 0.0
            
            # Sort by value descending
            results.sort(key=lambda x: x[col], reverse=True)
            return results
        
        finally:
            for h in handles:
                h.remove()

In [ ]:
#| export
class LayerProfiler:
    """Unified per-layer profiler for multiple metrics (speed, memory, size, compute)."""
    
    VALID_METRICS = frozenset({"speed", "memory", "size", "compute"})
    _CONFIG = {
        "speed": {
            "col": "speed_ms", "pct": "speed_percent", "unit": "ms",
            "label": "Speed (slowest)",
            "src_col": "time_ms",
            "format": _fmt_float,
        },
        "memory": {
            "col": "memory_mib", "pct": "memory_percent", "unit": "MiB",
            "label": "Memory (largest)",
            "src_col": "memory_mib",
            "format": _fmt_float,
        },
        "size": {
            "col": "params", "pct": "params_percent", "unit": "",
            "label": "Parameters (largest)",
            "src_col": "params",
            "format": _fmt_table,
        },
        "compute": {
            "col": "macs", "pct": "macs_percent", "unit": "",
            "label": "MACs (most)",
            "src_col": "macs",
            "format": _fmt_macs,
        },
    }
    
    def __init__(
        self,
        model: nn.Module,      # model to profile
        sample: torch.Tensor,  # input tensor (with batch dimension)
    ):
        self.model = model
        self.sample = sample
        self._leaf_modules = _leaf_modules(model)
        self._results: list[dict] = []
        self._profiled_metrics: list[str] = []
    
    def profile(
        self,
        metrics: str | Sequence[str] = "speed",  # metrics to profile: speed, memory, size, compute
        *,
        device: str | torch.device = "cpu",      # device for speed/memory profiling
        warmup: int = 5,                         # warmup iterations
        steps: int = 20,                         # measurement iterations
    ) -> list[dict]:
        """Profile specified metrics for each layer, returns list of dicts."""
        if isinstance(metrics, str):
            metrics = [metrics]
        metrics = list(metrics)
        
        invalid = set(metrics) - self.VALID_METRICS
        if invalid:
            raise ValueError(f"Invalid metrics: {invalid}. Valid: {self.VALID_METRICS}")
        
        # Initialize results dict for each leaf module
        results: dict[str, dict] = {
            name: {"name": name, "type": mod.__class__.__name__}
            for name, mod in self._leaf_modules.items()
        }
        
        # Profile each metric using unified _profile_layers
        for metric in metrics:
            cfg = self._CONFIG[metric]
            profile_data = _profile_layers(
                self.model, self.sample,
                device=device, metric=metric, warmup=warmup, steps=steps
            )
            
            # Merge results
            for r in profile_data:
                name = r["name"]
                if name in results:
                    results[name][cfg["col"]] = r[cfg["src_col"]]
                    results[name][cfg["pct"]] = r["percent"]
        
        out = list(results.values())
        
        # Sort by first metric
        sort_col = self._CONFIG[metrics[0]]["col"]
        out.sort(key=lambda x: x.get(sort_col, 0) or 0, reverse=True)
        
        # Store for top() and summary()
        self._results = out
        self._profiled_metrics = metrics
        
        return out
    
    def top(
        self,
        metric: str,              # metric to sort by: speed, memory, size, compute
        n: int = 5,               # number of layers to return
        *,
        ascending: bool = False,  # if True, return smallest/fastest instead of largest/slowest
    ) -> list[dict]:
        """Get top N layers sorted by the specified metric."""
        if not self._results:
            raise RuntimeError("No results available. Call profile() first.")
        
        if metric not in self._CONFIG:
            raise ValueError(f"Invalid metric: {metric}. Valid: {list(self._CONFIG.keys())}")
        
        col = self._CONFIG[metric]["col"]
        
        # Check if metric was profiled
        if col not in self._results[0]:
            raise ValueError(f"Metric '{metric}' was not profiled. Profiled: {self._profiled_metrics}")
        
        sorted_results = sorted(
            self._results,
            key=lambda x: x.get(col, 0) or 0,
            reverse=not ascending
        )
        return sorted_results[:n]
    
    def summary(self, *, top: int = 5) -> None:
        """Print a formatted summary of top layers for each profiled metric."""
        if not self._results:
            raise RuntimeError("No results available. Call profile() first.")
        
        for metric in self._profiled_metrics:
            cfg = self._CONFIG[metric]
            col, pct_col = cfg["col"], cfg["pct"]
            
            # Check if metric data exists
            if col not in self._results[0]:
                continue
            
            print(_section(cfg['label'], 54))
            
            sorted_layers = sorted(
                self._results,
                key=lambda x: x.get(col, 0) or 0,
                reverse=True
            )[:top]
            
            for r in sorted_layers:
                val = r.get(col, 0) or 0
                pct = r.get(pct_col, 0) or 0
                val_str = cfg["format"](val)
                if cfg["unit"]:
                    val_str += f" {cfg['unit']}"
                print(f"  {r['name']:40} {r['type']:15} {val_str} ({pct:5.1f}%)")
            print()